# gigapath_runpod (로컬→RunPod SSH 오케스트레이션)

로컬 주피터 노트북에서 SSH/rsync로 RunPod 인스턴스를 제어하며 **SVS 업로드 → 온더플라이 좌표 추출/벡터화 → 원본 삭제 → 학습** 순서로 스토리지 사용을 최소화합니다. PNG 타일을 남기지 않고 GPU에서 바로 임베딩만 저장합니다.


## 0. 기본 설정 (호스트/경로/커맨드 자리표시자)

- `SSH_HOST_DIRECT/PORT`: RunPod Direct TCP (SCP/rsync 가능)
- `SSH_HOST_GATEWAY`: 필요 시 게이트웨이 사용 (SCP 제한)
- `REMOTE_RAW`, `REMOTE_WORK`: RunPod 내부 경로
- 온더플라이 벡터화 커맨드 템플릿을 환경에 맞게 수정 (타일 PNG 미저장)
- 삭제 정책 토글: `DELETE_SVS_AFTER_FEATURE`
- 배치/타일/조직 필터 파라미터: `TILE_SIZE`, `MIN_TISSUE_RATIO`, `FEATURE_BATCH_SIZE`


In [ ]:
import os
import subprocess, shlex, time
from pathlib import Path

# SSH 대상 및 키 (실제 RunPod 접속 정보로 설정)
# 게이트웨이(ssh.runpod.io)와 Direct TCP(root@IP -p PORT)를 모두 지원
SSH_HOST_GATEWAY = 'uu70i6r3gdpk41-64410aaa@ssh.runpod.io'  # proxied, SCP 제한
SSH_HOST_DIRECT = 'root@216.81.245.80'  # direct TCP, SCP/rsync 가능
SSH_PORT_DIRECT = 40662
SSH_KEY = '~/.ssh/runpod_peter'  # 필요 없으면 None
# 기본은 Direct TCP를 사용(PTY 오류 회피, rsync/ssh 모두 동일 경로)
USE_DIRECT_FOR_SSH = True
USE_DIRECT_FOR_RSYNC = True
# PTY 옵션: direct에서는 비워두고, 게이트웨이 필요 시 설정 (예: '-T')
SSH_EXTRA_OPTS = ''

# RunPod 내부 경로
REMOTE_BASE = '/workspace/data'
REMOTE_RAW = f"{REMOTE_BASE}/raw"
REMOTE_WORK = f"{REMOTE_BASE}/work"
REMOTE_CHECKPOINT = f"{REMOTE_WORK}/checkpoints"
REMOTE_LOG = f"{REMOTE_BASE}/logs"

# 온더플라이 벡터화 설정
TILE_SIZE = 224
MIN_TISSUE_RATIO = 0.5
LEVEL = 0
FEATURE_BATCH_SIZE = 64
FEATURE_MODEL = 'hf_hub:prov-gigapath/prov-gigapath'

def _find_env_file(start: Path) -> Path | None:
    for p in [start] + list(start.parents):
        candidate = p / '.env'
        if candidate.exists():
            return candidate
    return None

def _load_hf_token():
    token = (
        os.environ.get('HUGGINGFACE_HUB_TOKEN')
        or os.environ.get('HUGGINGFACE_TOKEN')
        or os.environ.get('HF_HUB_TOKEN')
        or os.environ.get('HF_TOKEN')
    )
    if token:
        return token
    env_path = _find_env_file(Path.cwd())
    if env_path:
        for line in env_path.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            k, v = line.split('=', 1)
            if k.strip() in ('HUGGINGFACE_HUB_TOKEN', 'HUGGINGFACE_TOKEN', 'HF_HUB_TOKEN', 'HF_TOKEN'):
                return v.strip()
    return None

HF_TOKEN_VALUE = _load_hf_token()

# PNG 저장 없이 좌표 필터링→GPU 벡터화 템플릿
FEATURE_CMD_TEMPLATE = r"""
python - <<'PY'
import json
import time
from pathlib import Path

import numpy as np
import openslide
import torch
import timm
from PIL import Image
from skimage import color, filters
from timm.data import create_transform, resolve_model_data_config
import os
from huggingface_hub import login
token = (
    os.environ.get('HUGGINGFACE_HUB_TOKEN')
    or os.environ.get('HUGGINGFACE_TOKEN')
    or os.environ.get('HF_HUB_TOKEN')
    or os.environ.get('HF_TOKEN')
)
if token:
    try:
        login(token=token)
        print('huggingface login: token from env applied')
    except Exception as e:
        print(f'warning: huggingface login failed: {e}')
svs_path = Path('{input}').expanduser()
out_path = Path('{out}').expanduser()
tile_size = {tile_size}
min_tissue_ratio = {min_tissue_ratio}
level = {level}
batch_size = {batch_size}
model_name = '{model_name}'
if not svs_path.exists():
    raise SystemExit(f'SVS not found: {svs_path}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device != 'cuda':
    raise SystemExit('CUDA device not available for feature extraction')
torch.backends.cudnn.benchmark = True
slide = openslide.OpenSlide(str(svs_path))
width, height = slide.level_dimensions[level]
Image.MAX_IMAGE_PIXELS = None
def tissue_ratio(np_rgb):
    hsv = color.rgb2hsv(np_rgb)
    sat = hsv[:, :, 1]
    thresh = filters.threshold_otsu(sat)
    return float((sat > thresh).mean())
model = timm.create_model(model_name, pretrained=True).to(device)
model.eval()
config = resolve_model_data_config(model)
transform = create_transform(**config, is_training=False)
coords = []
feats = []
batch_imgs = []
batch_xy = []
log_every = 200
start = time.time()
with torch.inference_mode():
    for y in range(0, height, tile_size):
        for x in range(0, width, tile_size):
            region = slide.read_region((x, y), level, (tile_size, tile_size)).convert('RGB')
            arr = np.asarray(region)
            if arr.shape[0] < tile_size or arr.shape[1] < tile_size:
                continue
            if tissue_ratio(arr) < min_tissue_ratio:
                continue
            batch_imgs.append(transform(region))
            batch_xy.append((x, y))
            if len(batch_imgs) == batch_size:
                batch = torch.stack(batch_imgs).to(device, non_blocking=True)
                emb = model(batch)
                if isinstance(emb, (tuple, list)):
                    emb = emb[0]
                feats.append(emb.detach().cpu())
                coords.extend(batch_xy)
                batch_imgs.clear(); batch_xy.clear()
                if len(coords) % log_every == 0:
                    elapsed = time.time() - start
                    print(f'encoded {len(coords)} tiles | elapsed {elapsed/60:.1f}m')
    if batch_imgs:
        batch = torch.stack(batch_imgs).to(device, non_blocking=True)
        emb = model(batch)
        if isinstance(emb, (tuple, list)):
            emb = emb[0]
        feats.append(emb.detach().cpu())
        coords.extend(batch_xy)
slide.close()
if not coords:
    raise SystemExit('No tissue tiles passed threshold; check min_tissue_ratio/tile_size')
feat_tensor = torch.cat(feats, dim=0)
coord_tensor = torch.tensor(coords, dtype=torch.int32)
meta = {
    'slide': svs_path.name,
    'tile_size': tile_size,
    'level': level,
    'dims': {'width': width, 'height': height},
    'num_tiles': len(coords),
    'min_tissue_ratio': min_tissue_ratio,
    'batch_size': batch_size,
    'model': model_name,
    'created_at': time.time(),
}
out_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({'features': feat_tensor, 'coords': coord_tensor, 'meta': meta}, out_path)
size_mb = out_path.stat().st_size / (1024*1024)
print(f'saved {feat_tensor.shape} -> {out_path} ({size_mb:.1f} MB)')
print('total time: {:.1f} min'.format((time.time()-start)/60))
PY
"""

TRAIN_CMD_TEMPLATE = (
    'python train.py --features {features_dir} --out {ckpt_dir} --logdir {log_dir}'
)

# 삭제 정책
DELETE_SVS_AFTER_FEATURE = True

print('SSH_HOST_DIRECT=', SSH_HOST_DIRECT, 'port', SSH_PORT_DIRECT)
print('SSH_HOST_GATEWAY=', SSH_HOST_GATEWAY)
print('REMOTE_RAW=', REMOTE_RAW)
print('REMOTE_WORK=', REMOTE_WORK)
print('FEATURE_CMD_TEMPLATE=', FEATURE_CMD_TEMPLATE)
print('TRAIN_CMD_TEMPLATE=', TRAIN_CMD_TEMPLATE)


## 1. 유틸리티: 로컬 실행/SSH/rsync 래퍼

- `run_local`: 로컬 셸 실행
- `run_ssh`: RunPod에서 명령 실행
- `rsync_upload` / `rsync_download`: 부분 업로드/다운로드 지원


In [ ]:
def run_local(cmd, check=True):
    print(f"[local] $ {cmd}")
    # capture stdout/stderr so rsync/ssh 에러 메시지를 바로 확인
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        msg = result.stderr.strip() or result.stdout.strip()
        raise RuntimeError(f"Local command failed ({result.returncode}): {cmd}{msg}")
    return result.returncode


def _ssh_parts(host, port=None):
    parts = ["ssh"]
    if SSH_KEY:
        parts += ["-i", SSH_KEY]
    if SSH_EXTRA_OPTS:
        parts += shlex.split(SSH_EXTRA_OPTS)
    if port:
        parts += ["-p", str(port)]
    parts += [host]
    return parts


def run_ssh(cmd, check=True, use_direct=None):
    if use_direct is None:
        use_direct = USE_DIRECT_FOR_SSH
    parts = _ssh_parts(
        SSH_HOST_DIRECT if use_direct else SSH_HOST_GATEWAY,
        SSH_PORT_DIRECT if use_direct else None,
    )
    ssh_cmd = " ".join(shlex.quote(p) for p in parts + [cmd])
    print(f"[ssh] $ {cmd}")
    result = subprocess.run(ssh_cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        msg = result.stderr.strip() or result.stdout.strip()
        raise RuntimeError(f"SSH command failed ({result.returncode}): {cmd}{msg}")
    return result.returncode


def _rsync_ssh_opt(use_direct=True):
    # Build ssh transport for rsync (-e). Host is kept separate to avoid being injected into -e.
    key = Path(SSH_KEY).expanduser() if SSH_KEY else None
    parts = ["ssh"]
    if key:
        parts += ["-i", str(key)]
    if SSH_EXTRA_OPTS:
        parts += shlex.split(SSH_EXTRA_OPTS)
    if use_direct:
        parts += ["-p", str(SSH_PORT_DIRECT)]
    return " ".join(parts)


def rsync_upload(local_path: Path, remote_dir: str):
    use_direct = USE_DIRECT_FOR_RSYNC
    ssh_opt = _rsync_ssh_opt(use_direct)
    remote_host = SSH_HOST_DIRECT if use_direct else SSH_HOST_GATEWAY
    remote_target = f"{remote_host}:{remote_dir.rstrip('/')}/"
    cmd = (
        f"rsync -avh --no-perms --no-owner --no-group --partial --progress -e {shlex.quote(ssh_opt)} "
        f"{shlex.quote(str(local_path))} {remote_target}"
    )
    run_local(cmd)


def rsync_download(remote_path: str, local_dir: Path):
    use_direct = USE_DIRECT_FOR_RSYNC
    ssh_opt = _rsync_ssh_opt(use_direct)
    remote_host = SSH_HOST_DIRECT if use_direct else SSH_HOST_GATEWAY
    local_dir.mkdir(parents=True, exist_ok=True)
    remote_src = f"{remote_host}:{remote_path}"
    cmd = (
        f"rsync -avh --no-perms --no-owner --no-group --partial --progress -e {shlex.quote(ssh_opt)} "
        f"{remote_src} {shlex.quote(str(local_dir))}/"
    )
    run_local(cmd)


## 2. 원격 기본 디렉토리 준비 및 점검

- 필요한 폴더 생성
- GPU/디스크 확인 (실패 시에도 진행되도록 `|| true`)
- `ssh` 접속 테스트까지 포함


In [ ]:
# 접속 및 경로 생성
run_ssh(f"mkdir -p {REMOTE_RAW} {REMOTE_WORK} {REMOTE_CHECKPOINT} {REMOTE_LOG}")
run_ssh("echo 'SSH OK on $(hostname)' && nvidia-smi || true", check=False)
run_ssh("df -h . || true", check=False)


## 2.5 원격 의존성 설치 (openslide/timm 등)

- RunPod에 openslide/python 패키지가 없을 때 실행
- 최소 의존성: openslide-python, pillow, numpy, scikit-image, timm (torch는 미포함: 런팟 기본 CUDA 이미지 가정)


In [ ]:
REMOTE_PY_PKGS = ['openslide-python', 'Pillow', 'numpy', 'scikit-image', 'timm']
REMOTE_VENV = '~/venv_gigapath'

RUN_REMOTE_SETUP = True

def install_remote_deps(pkgs=None, venv_path=REMOTE_VENV):
    pkgs = pkgs or REMOTE_PY_PKGS
    pkg_str = ' '.join(pkgs)
    cmd = (
        f'python3 -m venv {venv_path} && ' +
        f'{venv_path}/bin/pip install --upgrade pip setuptools wheel && ' +
        f'{venv_path}/bin/pip install {pkg_str}'
    )
    print('[remote setup] installing into venv:', venv_path, '| pkgs:', pkg_str)
    run_ssh(cmd)

if RUN_REMOTE_SETUP:
    install_remote_deps()
else:
    print('Remote dependency install skipped (set RUN_REMOTE_SETUP=True)')


## 3. 데이터 선택: mammary adenoma/adenocarcinoma 100개씩 (외장 2023)

- `mammary_adenoma_vs_adenocarcinoma_only(2023).parquet`에서 라벨을 읽어 두 클래스만 필터링
- `/Volumes/Expansion/2023/<FOLDER>/<FILE_NAME>.svs` 존재 여부 확인 후 클래스별 100개씩 셔플 샘플링
- 선택 결과는 `selected_slides` 리스트에 `slide_id/path/label` 필드로 저장


In [ ]:
import random
import pandas as pd

SOURCE_SLIDE_ROOT = Path("/Volumes/Expansion/2023")
LABEL_PARQUET = Path("/Users/curv/Repos/GC-Pathology/PoC/v1/mammary_adenoma_vs_adenocarcinoma_only(2023).parquet")
SAMPLES_PER_CLASS = 100
LABEL_MAP = {"mammary_adenoma": 0, "mammary_adenocarcinoma": 1}

def _normalize_labels(df: pd.DataFrame) -> pd.DataFrame:
    label_norm = df["label"].astype(str).str.lower().str.strip()
    filtered = df.loc[label_norm.isin(LABEL_MAP.keys())].copy()
    filtered["label"] = label_norm.loc[filtered.index].map(LABEL_MAP)
    return filtered

def _collect(rows: pd.DataFrame, per_class: int):
    rows = rows.sample(frac=1, random_state=42).to_dict("records")
    picked = []
    for row in rows:
        slide_dir = SOURCE_SLIDE_ROOT / str(row["FOLDER"]).strip()
        file_names = [fn.strip() for fn in str(row["FILE_NAME"]).split("|") if fn.strip()]
        slide_id = row.get("INSP_RQST_NO", row.get("INSP_RQST_NUM", row["FOLDER"]))
        for file_name in file_names:
            fn = file_name if file_name.lower().endswith(".svs") else f"{file_name}.svs"
            slide_path = slide_dir / fn
            if slide_path.exists():
                picked.append({"slide_id": slide_id, "path": slide_path, "label": int(row["label"])})
                break
        if len(picked) >= per_class:
            break
    return picked

df_labels = _normalize_labels(pd.read_parquet(LABEL_PARQUET))
pos_rows = df_labels[df_labels["label"] == 1]
neg_rows = df_labels[df_labels["label"] == 0]
if pos_rows.empty or neg_rows.empty:
    raise ValueError(f"라벨 분포 확인 필요: pos={len(pos_rows)}, neg={len(neg_rows)}")

pos_pick = _collect(pos_rows, SAMPLES_PER_CLASS)
neg_pick = _collect(neg_rows, SAMPLES_PER_CLASS)
if len(pos_pick) < SAMPLES_PER_CLASS or len(neg_pick) < SAMPLES_PER_CLASS:
    print(f"경고: 요청 개수보다 부족 (pos {len(pos_pick)}/{SAMPLES_PER_CLASS}, neg {len(neg_pick)}/{SAMPLES_PER_CLASS})")

selected_slides = pos_pick[:SAMPLES_PER_CLASS] + neg_pick[:SAMPLES_PER_CLASS]
random.shuffle(selected_slides)

pos_cnt = sum(rec["label"] for rec in selected_slides)
neg_cnt = len(selected_slides) - pos_cnt
print(f"선정된 슬라이드: {len(selected_slides)}개 (pos={pos_cnt}, neg={neg_cnt})")
print("예시 3개:", selected_slides[:3])


## 4. 원격 온더플라이 벡터화 함수 (SVS 1개)

- RunPod에서 PNG 타일 저장 없이 즉시 임베딩 추출
- `FEATURE_CMD_TEMPLATE`(0단계) 사용, 결과는 `{stem}_gigapath.pt` 형식으로 저장(좌표/메타 포함)


In [ ]:
def remote_extract_features(svs_remote_path: str, batch_size: int = None) -> str:
    svs_name = Path(svs_remote_path).name
    out_path = f"{REMOTE_WORK}/{Path(svs_name).stem}_gigapath.pt"
    replacements = {
        'input': svs_remote_path,
        'out': out_path,
        'tile_size': TILE_SIZE,
        'min_tissue_ratio': MIN_TISSUE_RATIO,
        'level': LEVEL,
        'batch_size': batch_size or FEATURE_BATCH_SIZE,
        'model_name': FEATURE_MODEL,
    }
    cmd = FEATURE_CMD_TEMPLATE.strip()
    for key, val in replacements.items():
        cmd = cmd.replace(f'{{{key}}}', str(val))
    python_prefix = (REMOTE_VENV + '/bin/python ') if 'REMOTE_VENV' in globals() else 'python '
    cmd = cmd.replace("python - <<'PY'", python_prefix + "- <<'PY'")
    env_prefix = ''
    token = HF_TOKEN_VALUE if 'HF_TOKEN_VALUE' in globals() else None
    if token:
        env_prefix = (
            f"env HUGGINGFACE_HUB_TOKEN={token} HUGGINGFACE_TOKEN={token} HF_HUB_TOKEN={token} HF_TOKEN={token} "
        )
    run_ssh(env_prefix + cmd)
    if DELETE_SVS_AFTER_FEATURE:
        run_ssh(f"rm -f {svs_remote_path}")
        print(f"Deleted original svs: {svs_remote_path}")
    print(f"Feature saved: {out_path}")
    return out_path


## 5. 벡터 출력 포맷 (PNG 저장 없음)

- 저장: `torch.save({'features': Tensor[N, D], 'coords': int32[N,2], 'meta': {...}})`
- `meta`에는 slide 이름/타일 크기/level/원본 크기/모델명/필터 파라미터가 포함
- 필요시 `torch.load(..., map_location='cpu')`로 바로 사용


## 5.5 파라미터 튜닝 가이드

- `MIN_TISSUE_RATIO`를 높이면 조직 필터가 더 엄격해져 배경 타일이 줄어듭니다.
- `FEATURE_BATCH_SIZE`는 GPU 메모리에 맞춰 조정하세요 (예: 32/64/96).
- 타일 PNG 저장 로직은 제거되었으므로 디스크 I/O 병목 없이 곧바로 임베딩만 생성됩니다.


## 6. 선택 슬라이드 업로드→온더플라이 벡터화 (순차 처리)

- `selected_slides`(각 클래스 최대 100개)를 순서대로 업로드/처리
- 슬라이드 1개 단위로 rsync 업로드 → GPU에서 즉시 벡터화 → 옵션에 따라 원본 삭제
- `RUN_SLIDE_PROCESSING` 토글로 실행 여부를 제어하고, `PROCESS_LIMIT`로 일부만 시범 실행 가능


In [ ]:
def process_selected_slides(slides=None, limit=None, batch_size=None):
    slides = slides or selected_slides
    if not slides:
        print('selected_slides가 비어있습니다. 3단계 데이터 선택 셀을 실행하세요.')
        return []

    to_run = slides if limit is None else slides[:limit]
    results = []
    for idx, rec in enumerate(to_run, 1):
        local_path = Path(rec["path"])
        remote_svs = f"{REMOTE_RAW}/{local_path.name}"
        print(f"[{idx}/{len(to_run)}] {local_path.name} (label={rec['label']})")
        rsync_upload(local_path, REMOTE_RAW)
        start = time.time()
        feat_path = remote_extract_features(remote_svs, batch_size=batch_size)
        elapsed = time.time() - start
        results.append({
            **rec,
            "remote_svs": remote_svs,
            "feature_path": feat_path,
            "elapsed_min": elapsed / 60,
            "batch_size": batch_size or FEATURE_BATCH_SIZE,
        })
        print(f"완료: {local_path.name} -> {feat_path} ({elapsed/60:.1f} min)")
    return results

RUN_SLIDE_PROCESSING = True
PROCESS_LIMIT = None  # 예: 10 으로 설정하면 앞 10개만 처리
FEATURE_BATCH_OVERRIDE = None  # 예: 32

feature_results = []
if RUN_SLIDE_PROCESSING:
    feature_results = process_selected_slides(limit=PROCESS_LIMIT, batch_size=FEATURE_BATCH_OVERRIDE)
else:
    print('Processing skipped (RUN_SLIDE_PROCESSING=True 로 순차 실행)')


## 6.5 이미 업로드된 SVS 재벡터화/배치 처리 (옵션)

- raw 디렉토리에 있는 임의의 SVS 리스트를 다시 벡터화할 때 사용
- PNG 타일이 없으므로 슬라이드 경로만 전달하면 됨


In [ ]:
def vectorize_remote_svs(svs_paths, batch_size=None):
    if not svs_paths:
        print('svs_paths 가 비어있습니다. 예: ["/workspace/data/raw/sample.svs"]')
        return []
    results = []
    for idx, svs_path in enumerate(svs_paths, 1):
        print(f'[re-run {idx}/{len(svs_paths)}] {svs_path}')
        start = time.time()
        feat_path = remote_extract_features(svs_path, batch_size=batch_size)
        elapsed = time.time() - start
        results.append({
            'remote_svs': svs_path,
            'feature_path': feat_path,
            'elapsed_min': elapsed/60,
            'batch_size': batch_size or FEATURE_BATCH_SIZE,
        })
    return results

RUN_REMOTE_VECTORIZER = True
MANUAL_SVS_LIST = ["/workspace/data/raw/S23-00963#1###7.svs"]  # 예: [f"{REMOTE_RAW}/sample.svs"]
manual_vectorize_results = []
if RUN_REMOTE_VECTORIZER:
    manual_vectorize_results = vectorize_remote_svs(MANUAL_SVS_LIST)
else:
    print('Manual vectorization skipped (set RUN_REMOTE_VECTORIZER=True)')


## 7. 원격 학습 실행 (옵션)

- `RUN_TRAINING=True`로 토글
- `TRAIN_CMD_TEMPLATE`를 gigapath 학습 스크립트에 맞게 수정
- 체크포인트/로그 경로는 REMOTE_CHECKPOINT/REMOTE_LOG


In [ ]:
RUN_TRAINING = False  # True로 바꾸면 학습 실행

def run_remote_training():
    train_cmd = TRAIN_CMD_TEMPLATE.format(
        features_dir=shlex.quote(REMOTE_WORK),
        ckpt_dir=shlex.quote(REMOTE_CHECKPOINT),
        log_dir=shlex.quote(REMOTE_LOG),
    )
    print(f"Training command:{train_cmd}")
    run_ssh(train_cmd)

if RUN_TRAINING:
    run_remote_training()
else:
    print("Training skipped (set RUN_TRAINING=True to run)")


## 8. 결과 다운로드/아카이브

- 필요 체크포인트/feature를 로컬로 rsync 다운로드
- 오래된 체크포인트는 RunPod에서 압축 후 삭제


In [ ]:
def download_checkpoints(local_dir: Path):
    rsync_download(f"{REMOTE_CHECKPOINT}/", local_dir)

def remote_archive_and_delete(path_glob: str):
    # 예: path_glob="~/data/work/checkpoints/epoch*"
    cmd = (
        "for p in {path_glob}; do [ -e \"$p\" ] || continue; "
        "tar -czf ${p}.tar.gz -C $(dirname \"$p\") $(basename \"$p\") && rm -rf \"$p\"; done"
    ).replace("{path_glob}", path_glob)
    run_ssh(cmd)


## 9. 추천 워크플로우 (셀 실행 순서)

1) 0~2 셀 실행: 설정/접속/경로 준비
2) 2.5 셀로 원격 의존성 설치 필요 시 실행(RUN_REMOTE_SETUP=True)
3) 3 셀 실행: 외장 `/Volumes/Expansion/2023`에서 클래스별 100개 슬라이드 샘플링(`selected_slides`)
4) 4~5.5 셀 실행: 온더플라이 벡터화 함수/파라미터 확인(FEATURE_CMD_TEMPLATE 포함)
5) 6 셀에서 `RUN_SLIDE_PROCESSING=True` 설정 후 실행: 업로드→GPU 벡터화(타일 PNG 없음)
6) 필요 시 7 셀에서 학습 실행, 8 셀로 결과 다운로드/정리

공간 절약: 원본 삭제만 선택적으로 수행하며, 별도의 타일 PNG를 만들지 않으므로 디스크 병목 없이 빠르게 임베딩을 생성합니다.
